In [ ]:
!pip install kagglehub pandas

import kagglehub
import os
import pandas as pd

path = kagglehub.dataset_download("shivamb/go-emotions-google-emotions-dataset")
print("Path to dataset files:", path)

csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]

if csv_files:
    csv_path = os.path.join(path, csv_files[0])
    df = pd.read_csv(csv_path)
    print(f"\n✅ Dataset loaded successfully in Colab! Shape: {df.shape}")
    print(df.head())

Using Colab cache for faster access to the 'go-emotions-google-emotions-dataset' dataset.
Path to dataset files: /kaggle/input/go-emotions-google-emotions-dataset

✅ Dataset loaded successfully in Colab! Shape: (211225, 31)
        id                                               text  \
0  eew5j0j                                    That game hurt.   
1  eemcysk   >sexuality shouldn’t be a grouping category I...   
2  ed2mah1     You do right, if you don't care then fuck 'em!   
3  eeibobj                                 Man I love reddit.   
4  eda6yn6  [NAME] was nowhere near them, he was by the Fa...   

   example_very_unclear  admiration  amusement  anger  annoyance  approval  \
0                 False           0          0      0          0         0   
1                  True           0          0      0          0         0   
2                 False           0          0      0          0         0   
3                 False           0          0      0          0         

## Step 1: Data Import & Sanity Inspection

In [ ]:
pip install datasets

In [ ]:
import pandas as pd

# The dataset has already been loaded into 'df' from the KaggleHub download.
# We will now perform the requested sanity checks on this existing DataFrame.

print(f"Dataset 'df' is already loaded from KaggleHub.")
print(f"Shape of DataFrame: {df.shape}")
print("\nColumns in the DataFrame:")
for col in df.columns:
    print(f"- {col}")

print("\nMissing values per column:")
print(df.isnull().sum())

print(f"\nTotal raw rows: {len(df)}")
print("\nFirst 5 rows of the DataFrame:")
display(df.head())

Dataset 'df' is already loaded from KaggleHub.
Shape of DataFrame: (211225, 31)

Columns in the DataFrame:
- id
- text
- example_very_unclear
- admiration
- amusement
- anger
- annoyance
- approval
- caring
- confusion
- curiosity
- desire
- disappointment
- disapproval
- disgust
- embarrassment
- excitement
- fear
- gratitude
- grief
- joy
- love
- nervousness
- optimism
- pride
- realization
- relief
- remorse
- sadness
- surprise
- neutral

Missing values per column:
id                      0
text                    0
example_very_unclear    0
admiration              0
amusement               0
anger                   0
annoyance               0
approval                0
caring                  0
confusion               0
curiosity               0
desire                  0
disappointment          0
disapproval             0
disgust                 0
embarrassment           0
excitement              0
fear                    0
gratitude               0
grief                   0
joy  

,id,text,example_very_unclear,admiration,amusement,anger,annoyance,approval,caring,confusion,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,eew5j0j,That game hurt.,False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,eemcysk,>sexuality shouldn’t be a grouping category I...,True,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,ed2mah1,"You do right, if you don't care then fuck 'em!",False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,eeibobj,Man I love reddit.,False,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
4,eda6yn6,"[NAME] was nowhere near them, he was by the Fa...",False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


## Step 2: Mapping & Filtering

In [ ]:
import numpy as np

# Define the mapping from original emotion columns to target class IDs
emotion_to_target = {
    'neutral': 0,
    'sadness': 1,
    'grief': 1,
    'joy': 2,
    'amusement': 2,
    'excitement': 2,
    'optimism': 2,
    'anger': 3,
    'annoyance': 3,
    'disapproval': 3,
    'disgust': 3,
    'desire': 4,
    'fear': 5,
    'nervousness': 5,
    'love': 6
}

# Group the original emotion columns by target class ID
target_id_to_emotions = {
    0: ['neutral'],
    1: ['sadness', 'grief'],
    2: ['joy', 'amusement', 'excitement', 'optimism'],
    3: ['anger', 'annoyance', 'disapproval', 'disgust'],
    4: ['desire'],
    5: ['fear', 'nervousness'],
    6: ['love']
}

# Prioritized list of target IDs for single-label assignment (higher index = higher priority)
# This is a heuristic to resolve cases where multiple primary emotions might be present.
# For example, 'love' might override 'joy'. Adjust this order if a different prioritization is desired.
PRIORITY_TARGET_IDS = [0, 1, 2, 3, 4, 5, 6]

def assign_single_label(row):
    active_labels = []
    for target_id in PRIORITY_TARGET_IDS:
        relevant_emotions = target_id_to_emotions[target_id]
        # Check if any of the emotions contributing to this target_id are active in the row
        if any(row[col] == 1 for col in relevant_emotions if col in row.index):
            active_labels.append(target_id)

    if len(active_labels) == 1:
        return active_labels[0]
    elif len(active_labels) > 1:
        # If multiple target categories are active, pick the one with highest priority
        # defined by PRIORITY_TARGET_IDS order. The last one appended has the highest priority.
        return active_labels[-1]
    else:
        # If no specific target emotion is active, but neutral is, assign neutral
        if row['neutral'] == 1 and all(row[col] == 0 for col in df.columns if col not in ['id', 'text', 'example_very_unclear', 'neutral'] and col in row.index):
            return 0
        # If no target category is active, or 'example_very_unclear' is True, return NaN to indicate dropping
        if row['example_very_unclear']:
            return np.nan
        return np.nan

# Apply the function to create the 'encoded_label' column
initial_rows = len(df)
print(f"Initial number of rows: {initial_rows}")
df['encoded_label'] = df.apply(assign_single_label, axis=1)

# Drop rows where 'encoded_label' is NaN (i.e., not assigned to a target category)
df.dropna(subset=['encoded_label'], inplace=True)

# Also drop rows with empty or whitespace-only text
df = df[df['text'].str.strip() != ''].copy()

# Convert 'encoded_label' to integer type
df['encoded_label'] = df['encoded_label'].astype(int)

# Reset index after dropping rows
df.reset_index(drop=True, inplace=True)

remaining_rows = len(df)
print(f"Remaining rows after mapping and filtering: {remaining_rows}")
print(f"Dropped {initial_rows - remaining_rows} rows.")

print("\nValue count of each target class:")
print(df['encoded_label'].value_counts().sort_index())
print("\nPercentage distribution of each target class:")
print(df['encoded_label'].value_counts(normalize=True).sort_index() * 100)

print("\nFirst 5 rows of the filtered DataFrame with new labels:")
display(df.head())

Initial number of rows: 211225
Remaining rows after mapping and filtering: 137922
Dropped 73303 rows.

Value count of each target class:
encoded_label
0    55298
1     5962
2    26807
3    33579
4     3621
5     4464
6     8191
Name: count, dtype: int64

Percentage distribution of each target class:
encoded_label
0    40.093676
1     4.322733
2    19.436348
3    24.346370
4     2.625397
5     3.236612
6     5.938864
Name: proportion, dtype: float64

First 5 rows of the filtered DataFrame with new labels:


,id,text,example_very_unclear,admiration,amusement,anger,annoyance,approval,caring,confusion,...,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral,encoded_label
0,eew5j0j,That game hurt.,False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,1
1,ed2mah1,"You do right, if you don't care then fuck 'em!",False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
2,eeibobj,Man I love reddit.,False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,6
3,eda6yn6,"[NAME] was nowhere near them, he was by the Fa...",False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
4,eczuekb,"He isn't as big, but he's still quite popular....",False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,3


## Step 3: Text Cleaning & Deduplication

In [ ]:
import numpy as np

# Remove duplicate text entries
initial_rows = len(df)
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
print(f"Rows after removing duplicate text entries: {len(df)}")

# Strip unnecessary whitespace
df['text'] = df['text'].str.strip()

# Fix broken encodings (simple approach: coerce to utf-8 and remove non-printable characters)
# Note: A more robust solution might involve `ftfy` or `chardet` for complex cases.
df['text'] = df['text'].apply(lambda x: x.encode('utf-8', 'ignore').decode('utf-8'))

# Remove empty strings after cleaning
df = df[df['text'].str.strip() != ''].reset_index(drop=True)

remaining_rows = len(df)
print(f"Total clean rows after all cleaning steps: {remaining_rows}")
print(f"Dropped {initial_rows - remaining_rows} rows due to text deduplication or becoming empty after cleaning.")

# Calculate summary statistics
df['char_length'] = df['text'].apply(len)
# Approximate token length (very rough, as actual tokenization depends on the tokenizer)
# Using word count as a proxy for token count
df['token_length_approx'] = df['text'].apply(lambda x: len(x.split()))

print("\nSummary Statistics for Clean Text:")
print(f"- Average character length: {df['char_length'].mean():.2f}")
print(f"- Min character length: {df['char_length'].min()}")
print(f"- Max character length: {df['char_length'].max()}")
print(f"- Average approximate token length: {df['token_length_approx'].mean():.2f}")
print(f"- Min approximate token length: {df['token_length_approx'].min()}")
print(f"- Max approximate token length: {df['token_length_approx'].max()}")

print("\nFirst 5 rows of the cleaned DataFrame:")
display(df.head())

Rows after removing duplicate text entries: 51550
Total clean rows after all cleaning steps: 51550
Dropped 86372 rows due to text deduplication or becoming empty after cleaning.

Summary Statistics for Clean Text:
- Average character length: 69.97
- Min character length: 2
- Max character length: 542
- Average approximate token length: 13.15
- Min approximate token length: 1
- Max approximate token length: 33

First 5 rows of the cleaned DataFrame:


,id,text,example_very_unclear,admiration,amusement,anger,annoyance,approval,caring,confusion,...,pride,realization,relief,remorse,sadness,surprise,neutral,encoded_label,char_length,token_length_approx
0,eew5j0j,That game hurt.,False,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,15,3
1,ed2mah1,"You do right, if you don't care then fuck 'em!",False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,46,10
2,eeibobj,Man I love reddit.,False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,6,18,4
3,eda6yn6,"[NAME] was nowhere near them, he was by the Fa...",False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,51,10
4,eczuekb,"He isn't as big, but he's still quite popular....",False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,3,115,21


## Step 3: Fast Text Cleaning & Deduplication (Revised)

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
import os

# Keep original DataFrame for comparison in case of issues, though 'df' will be modified
# original_df_step2 = df.copy()

print(f"Initial rows before revised Step 3 cleaning: {len(df)}")

# 1. Apply basic string cleanup to the text column
df['text'] = df['text'].astype(str).str.strip()

# 2. Drop empty strings and duplicates
initial_rows_before_dedupe = len(df)
df = df[df['text'] != ""].drop_duplicates(subset=['text']).reset_index(drop=True)

cleaned_rows = len(df)
print(f"Total cleaned rows remaining after fast cleaning and deduplication: {cleaned_rows}")
print(f"Dropped {initial_rows_before_dedupe - cleaned_rows} rows due to empty text or duplicates.")

# 3. Calculate and print summary statistics
df['char_length'] = df['text'].str.len()
df['token_length_approx'] = df['text'].str.split().str.len()

print("\nSummary Statistics for Clean Text:")
print(f"- Average character length: {df['char_length'].mean():.2f}")
print(f"- Min character length: {df['char_length'].min()}")
print(f"- Max character length: {df['char_length'].max()}")

# Handle potential NaN from empty strings if any crept in (though we filtered above)
df['token_length_approx'].fillna(0, inplace=True)
print(f"- Average approximate token length: {df['token_length_approx'].mean():.2f}")
print(f"- Min approximate token length: {int(df['token_length_approx'].min())}")
print(f"- Max approximate token length: {int(df['token_length_approx'].max())}")

print("\nFirst 5 rows of the cleaned DataFrame:")
display(df.head())

Initial rows before revised Step 3 cleaning: 51550
Total cleaned rows remaining after fast cleaning and deduplication: 51549
Dropped 1 rows due to empty text or duplicates.

Summary Statistics for Clean Text:
- Average character length: 69.97
- Min character length: 2
- Max character length: 542
- Average approximate token length: 13.15
- Min approximate token length: 1
- Max approximate token length: 33

First 5 rows of the cleaned DataFrame:


/tmp/ipykernel_28491/2034742845.py:31: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['token_length_approx'].fillna(0, inplace=True)


,id,text,example_very_unclear,admiration,amusement,anger,annoyance,approval,caring,confusion,...,pride,realization,relief,remorse,sadness,surprise,neutral,encoded_label,char_length,token_length_approx
0,eew5j0j,That game hurt.,False,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,15,3
1,ed2mah1,"You do right, if you don't care then fuck 'em!",False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,46,10
2,eeibobj,Man I love reddit.,False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,6,18,4
3,eda6yn6,"[NAME] was nowhere near them, he was by the Fa...",False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,51,10
4,eczuekb,"He isn't as big, but he's still quite popular....",False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,3,115,21


## Step 4: Stratified Splitting & Export

In [ ]:
import os
from sklearn.model_selection import train_test_split

# Define target column for stratification
label_column = 'encoded_label'

# Ensure the 'saved_datasets' directory exists
os.makedirs('saved_datasets', exist_ok=True)

print(f"Initial number of rows for splitting: {len(df)}")
print(f"Target column for stratification: '{label_column}'")
print("Initial class distribution:")
print(df[label_column].value_counts(normalize=True).sort_index() * 100)

# 1. First split: 80% train, 20% temp (for val/test)
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df[label_column]
)

# Second split: 10% validation, 10% test from the 20% temp_df
# (0.5 of 20% is 10%)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df[label_column]
)

# 2. Reset indexes for all splits
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\n✅ Data Split Complete with Stratification and Index Reset:")
print(f" - Train set : {len(train_df)} samples")
print(f" - Validation set: {len(val_df)} samples")
print(f" - Test set  : {len(test_df)} samples")

# 3. Export datasets to CSV
train_df.to_csv('saved_datasets/train.csv', index=False)
val_df.to_csv('saved_datasets/validation.csv', index=False)
test_df.to_csv('saved_datasets/test.csv', index=False)

print("\n✅ Datasets exported to 'saved_datasets/' directory:")
print(" - saved_datasets/train.csv")
print(" - saved_datasets/validation.csv")
print(" - saved_datasets/test.csv")

# 4. Verify class balance across splits
print("\n--- Class Balance Verification ---")
print("\nTrain set class distribution:")
print(train_df[label_column].value_counts(normalize=True).sort_index() * 100)

print("\nValidation set class distribution:")
print(val_df[label_column].value_counts(normalize=True).sort_index() * 100)

print("\nTest set class distribution:")
print(test_df[label_column].value_counts(normalize=True).sort_index() * 100)

print("\n🎉 Step 4 Complete! All datasets are ready.")

Initial number of rows for splitting: 51549
Target column for stratification: 'encoded_label'
Initial class distribution:
encoded_label
0    42.074531
1     4.337621
2    20.039186
3    22.275893
4     2.490834
5     2.979689
6     5.802246
Name: proportion, dtype: float64

✅ Data Split Complete with Stratification and Index Reset:
 - Train set : 41239 samples
 - Validation set: 5155 samples
 - Test set  : 5155 samples

✅ Datasets exported to 'saved_datasets/' directory:
 - saved_datasets/train.csv
 - saved_datasets/validation.csv
 - saved_datasets/test.csv

--- Class Balance Verification ---

Train set class distribution:
encoded_label
0    42.074250
1     4.338127
2    20.039283
3    22.275031
4     2.490361
5     2.980189
6     5.802760
Name: proportion, dtype: float64

Validation set class distribution:
encoded_label
0    42.075655
1     4.325897
2    20.038797
3    22.289040
4     2.483026
5     2.987391
6     5.800194
Name: proportion, dtype: float64

Test set class distribution:

## Complete RoBERTa Fine-tuning Pipeline

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from datasets import Dataset  # Import Dataset from the datasets library
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# --- Global Configurations ---
MODEL_NAME = "roberta-base"
MAX_LENGTH = 128
NUM_LABELS = 7  # As per the 7 target categories
SAVE_DIR = "./saved_emotion_model"

# --- 1. Data Loading & Formatting ---
print("--- Step 1: Data Loading & Formatting ---")

# Load datasets from CSVs
train_df = pd.read_csv('saved_datasets/train.csv')
val_df = pd.read_csv('saved_datasets/validation.csv')
test_df = pd.read_csv('saved_datasets/test.csv')

# Convert Pandas DataFrames to Hugging Face Dataset objects
# We only need 'text' and 'encoded_label' columns
train_dataset_hf = Dataset.from_pandas(train_df[['text', 'encoded_label']])
val_dataset_hf = Dataset.from_pandas(val_df[['text', 'encoded_label']])
test_dataset_hf = Dataset.from_pandas(test_df[['text', 'encoded_label']])

print(f"Loaded train dataset: {len(train_dataset_hf)} samples")
print(f"Loaded validation dataset: {len(val_dataset_hf)} samples")
print(f"Loaded test dataset: {len(test_dataset_hf)} samples")

# --- 2. Tokenizer & Model Initialization ---
print("\n--- Step 2: Tokenizer & Model Initialization ---")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Reconstruct id2label and label2id based on the 7 target categories
# This needs to be consistent with how `encoded_label` was created in Step 2.
# 0: Neutral (`neutral`)
# 1: Sadness (`sadness`, `grief`)
# 2: Joy (`joy`, `amusement`, `excitement`, `optimism`)
# 3: Hate / Anger (`anger`, `annoyance`, `disapproval`, `disgust`)
# 4: Sexual / Desire (`desire`)
# 5: Fear / Anxiety (`fear`, `nervousness`)
# 6: Love (`love`)

id2label = {
    0: "neutral",
    1: "sadness_grief",
    2: "joy_amusement_excitement_optimism",
    3: "anger_annoyance_disapproval_disgust",
    4: "desire",
    5: "fear_nervousness",
    6: "love"
}
label2id = {v: k for k, v in id2label.items()}

# Define tokenization function
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding='max_length', max_length=MAX_LENGTH)

# Apply tokenization to all datasets
tokenized_train_dataset = train_dataset_hf.map(tokenize_function, batched=True)
tokenized_val_dataset = val_dataset_hf.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset_hf.map(tokenize_function, batched=True)

# Rename 'encoded_label' to 'labels' for Trainer compatibility
tokenized_train_dataset = tokenized_train_dataset.rename_column("encoded_label", "labels")
tokenized_val_dataset = tokenized_val_dataset.rename_column("encoded_label", "labels")
tokenized_test_dataset = tokenized_test_dataset.rename_column("encoded_label", "labels")

# Set format for PyTorch
tokenized_train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"]) # Removed 'id' as it's not needed for model input
tokenized_val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"]) # Removed 'id'
tokenized_test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"]) # Removed 'id'

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Tokenizer and {MODEL_NAME} model initialized and moved to {device}.")
print(f"Model config: num_labels={model.config.num_labels}, id2label={model.config.id2label}")

# Check if CUDA is available and warn if not
if not torch.cuda.is_available():
    print("\n=========================================================================================")
    print("WARNING: CUDA (GPU) is NOT available. Training will proceed on CPU, which will be much")
    print("slower and may lead to Out-Of-Memory errors for large models/batches. \n")
    print("To enable GPU, go to 'Runtime' -> 'Change runtime type' in the Colab menu and select 'T4 GPU' (or another available GPU). Then re-run all cells.")
    print("=========================================================================================")


# --- 3. Evaluation Metrics Function ---
print("\n--- Step 3: Evaluation Metrics Function ---")

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    accuracy = accuracy_score(p.label_ids, preds)
    # Use 'weighted' average to account for class imbalance
    precision, recall, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average='weighted', zero_division=0)
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

print("Custom `compute_metrics` function defined.")

# --- 4. Hugging Face Trainer Setup ---
print("\n--- Step 4: Hugging Face Trainer Setup ---")

# Data Collator for dynamic padding (will pad to max_length if not already)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Ensure GPU is available for fp16
if not torch.cuda.is_available():
    print("Warning: CUDA not available. fp16 training will be disabled.")
    fp16_enabled = False
else:
    fp16_enabled = True

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=32,  # User specified
    per_device_eval_batch_size=32,   # User specified
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",  # User specified
    greater_is_better=True,
    logging_dir='./logs',
    logging_steps=50,
    fp16=fp16_enabled,               # User specified
    gradient_accumulation_steps=1,
    report_to="none"  # Disable reporting to external services like W&B if not needed
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    tokenizer=tokenizer,  # Pass tokenizer to Trainer for DataCollatorWithPadding to work correctly
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Hugging Face Trainer initialized. Starting training...")
# Train the model
trainer.train()

print("\n🎉 Model Training Completed Successfully with Hugging Face Trainer!")

# --- 5. Final Evaluation & Inference Test ---
print("\n--- Step 5: Final Evaluation & Inference Test ---")

# Evaluate the best model on the test set
print("\nEvaluating best model on test set...")
test_metrics = trainer.evaluate(eval_dataset=tokenized_test_dataset)
print(f"Test Metrics: {test_metrics}")

# Create the directory for saving the model if it doesn't exist
os.makedirs(SAVE_DIR, exist_ok=True)

# Save the best model and tokenizer locally
trainer.save_model(SAVE_DIR)  # Saves both model and tokenizer by default
# The tokenizer is usually saved by save_model, but explicitly saving ensures all files are there
tokenizer.save_pretrained(SAVE_DIR)

print(f"✅ Trained model & tokenizer saved at '{SAVE_DIR}'!")

# Run 3 sample inferences
print("\n--- Sample Inferences ---")
# Load the saved model and tokenizer into a pipeline for inference
inference_pipeline = pipeline(
    "text-classification",
    model=SAVE_DIR,
    tokenizer=SAVE_DIR,
    device=0 if torch.cuda.is_available() else -1  # Use GPU if available (device=0), else CPU (-1)
)

sample_sentences = [
    "I love this product!",
    "I feel so lonely and hopeless.",
    "Stay away from me!"
]

for i, sentence in enumerate(sample_sentences):
    result = inference_pipeline(sentence)
    print(f"Sentence {i+1}: '{sentence}'")
    # The pipeline output label will be the string from id2label, e.g., 'love'
    print(f"Prediction: {result[0]['label']} (Score: {result[0]['score']:.4f})")
    print("-" * 30)

print("\n🎉 All tasks completed!")

ModuleNotFoundError: Could not import module 'TrainingArguments'. Are this object's requirements defined correctly?

# Step 1 - Exploratory Data Analysis (EDA) & Data Audit

In [ ]:
import numpy as np

# Cell 2: Audit dataset structure, missing values, and data types
print("--- DATASET AUDIT ---")

print("\n1. Data Types & Info:")
df.info()

print("\n2. describe:")
df.describe()

print("\n3. Missing Values Check:")
print(df.isnull().sum())

# Identify Text and Label Column Names dynamically
# FIX: Explicitly set text_col and label_col for this classification task.
text_col = 'text_transcript'
label_col = 'human_verified_label'

print(f"\n💡 Detected Text Column: '{text_col}' | Target Label Column: '{label_col}'")

## Step 1: Data Cleaning & Label Sanitization
## Step 2: Stratified Split & Strict Index Reset

In [ ]:
import re
import pandas as pd
from sklearn.model_selection import train_test_split

# --- Step 1: Raw Data Loading (using existing df) & Label Sanitization ---

initial_count = len(df)

# Drop duplicates based on text_col and then drop NaNs on text_col and label_col
df = df.drop_duplicates(subset=[text_col]).dropna(subset=[text_col, label_col]).reset_index(drop=True)

print(f"✅ Duplicate and NaN rows removed. Initial: {initial_count}, Final: {len(df)}")

# Ensure 'encoded_label' is explicitly mapped to clean binary integers (0 and 1) with int64 dtype
df['encoded_label'] = df[label_col].astype(int)

print(f"✅ '{label_col}' mapped to 'encoded_label' as int64. Unique values: {df['encoded_label'].unique()}")

# Verify balanced targets by printing value_counts
print("\n--- Encoded Label Distribution ---")
class_counts = df['encoded_label'].value_counts()
class_percentages = df['encoded_label'].value_counts(normalize=True) * 100

for cls_name, count in class_counts.items():
    print(f"Class '{cls_name}': {count} samples ({class_percentages[cls_name]:.2f}%)")

# --- Step 2: Stratified Split & Strict Index Reset ---

# Perform a stratified split (Train/Validation/Test split)
# The instructions asked for train/val only, but the Trainer setup uses a test set, so we'll create it.
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['encoded_label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['encoded_label']
)

# Call .reset_index(drop=True) on all dataframes
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\n✅ Data Split Complete with Stratification and Index Reset:")
print(f" - Train set : {len(train_df)} samples")
print(f" - Val set   : {len(val_df)} samples")
print(f" - Test set  : {len(test_df)} samples")

# Extract features and targets into pure Python lists
train_texts = train_df[text_col].astype(str).tolist()
train_labels = train_df['encoded_label'].astype(int).tolist()
val_texts = val_df[text_col].astype(str).tolist()
val_labels = val_df['encoded_label'].astype(int).tolist()
test_texts = test_df[text_col].astype(str).tolist()
test_labels = test_df['encoded_label'].astype(int).tolist()

print("✅ Features and targets extracted into Python lists.")

## Step 3: Baseline Sanity Check (TF-IDF + Logistic Regression)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

print("🚀 Running TF-IDF + Logistic Regression Baseline...")

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limiting features for speed

# Fit on training data and transform train and validation texts
X_train_tfidf = tfidf_vectorizer.fit_transform(train_texts)
X_val_tfidf = tfidf_vectorizer.transform(val_texts)

# Initialize and train Logistic Regression model
log_reg_model = LogisticRegression(max_iter=1000, random_state=42)
log_reg_model.fit(X_train_tfidf, train_labels)

# Make predictions on the validation set
y_pred_baseline = log_reg_model.predict(X_val_tfidf)

# Print classification report
print("\n--- Baseline Classification Report (Validation Set) ---")
print(classification_report(val_labels, y_pred_baseline, zero_division=0))

print("✅ Baseline model execution complete. Please check the report for >80% accuracy.")

## Tokenizer Initialization (from original Step 7)

In [ ]:
## Step 4: Fixed PyTorch Dataset Class

# Original Cell 6: Encode text categories into numeric IDs and save mapping
import json
import os

# The encoded_label is already set in the cleaning cell.
# Here we just generate the label mappings and save them.

unique_labels = sorted(df[label_col].unique().tolist())
# Ensure labels are treated as strings for the map keys for consistency if not already
label2id = {str(label): i for i, label in enumerate(unique_labels)}
id2label = {i: str(label) for i, label in enumerate(unique_labels)}

print("✅ Label Mapping Created:")
print(json.dumps(label2id, indent=2))

# Save mapping for FastAPI Inference later
os.makedirs("saved_models", exist_ok=True)
with open("saved_models/label_map.json", "w") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f, indent=4)

In [ ]:
# Cell 6: Encode text categories into numeric IDs and save mapping
import json

unique_labels = sorted(df[label_col].unique().tolist())
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for i, label in enumerate(unique_labels)}

df['encoded_label'] = df[label_col].map(label2id)

print("✅ Label Mapping Created:")
print(json.dumps(label2id, indent=2))

# Save mapping for FastAPI Inference later
os.makedirs("saved_models", exist_ok=True)
with open("saved_models/label_map.json", "w") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f, indent=4)

## Tokenizer Loading & MAX_LENGTH (from original Step 7)

In [ ]:
# Cell 7: Load RoBERTa Tokenizer and determine optimal max_length
import numpy as np
from transformers import AutoTokenizer

# Switched to roberta-base tokenizer
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

print("Calculating token lengths...")
token_lengths = [len(tokenizer.encode(str(t), truncation=False)) for t in df[text_col].iloc[:5000]] # Sample 5k for fast check

p95 = np.percentile(token_lengths, 95)
p99 = np.percentile(token_lengths, 99)

print(f"95th Percentile Token Length: {p95:.1f}")
print(f"99th Percentile Token Length: {p99:.1f}")

# MAX_LENGTH typically needs to be adjusted for RoBERTa. Setting to 128 as a good balance.
MAX_LENGTH = 128

print(f"💡 Selected MAX_LENGTH for RoBERTa-base: {MAX_LENGTH}")

# Sanity check for tokenization
print("\n--- Tokenization Sanity Check ---")
sample_text = df[text_col].iloc[0] # Corrected: access text directly from the DataFrame
sample_tokenized = tokenizer(sample_text, truncation=True, padding='max_length', max_length=MAX_LENGTH, return_tensors='pt')
print(f"Sample Original Text: {sample_text[:150]}...")
print(f"Sample Input IDs (first 10): {sample_tokenized['input_ids'][0, :10].tolist()}...")
print(f"Sample Decoded Tokens (first 10): {tokenizer.decode(sample_tokenized['input_ids'][0, :10]).replace(' ', ' ')}...")

if (sample_tokenized['input_ids'][0] == tokenizer.pad_token_id).all():
    print("⚠️ Warning: The first sample seems to be entirely padding tokens after tokenization. MAX_LENGTH might be too small or text is empty.")
else:
    print("✅ Tokenization check passed: Input IDs are not all padding and text is decoded correctly.")

## PyTorch Dataset Implementation

# Cell 9: Step 8 - PyTorch Dataset Abstraction

In [ ]:
import torch
from torch.utils.data import Dataset

class SafeTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        # Tokenize with truncation, padding, and return as Python lists
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors=None # Crucial: return as Python lists of integers
        )

        return {
            'input_ids': torch.tensor(encoding['input_ids'], dtype=torch.long),         # Shape: [max_length]
            'attention_mask': torch.tensor(encoding['attention_mask'], dtype=torch.long), # Shape: [max_length]
            'labels': torch.tensor(label, dtype=torch.long)                 # Scalar Tensor
        }

# Instantiate PyTorch Datasets with the new class
train_dataset = SafeTextDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
val_dataset = SafeTextDataset(val_texts, val_labels, tokenizer, MAX_LENGTH)
test_dataset = SafeTextDataset(test_texts, test_labels, tokenizer, MAX_LENGTH)

print("✅ SafeTextDataset PyTorch Datasets created successfully from lists!")

## DataLoader Creation & Pipeline Sanity Check (from original Step 10)

In [ ]:
# Cell 10: Create DataLoaders and run final Sanity Check on GPU tensors
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
import numpy as np

BATCH_SIZE = 16  # User specified: Adjusted batch size to 16

# Initialize Data Collator for dynamic padding, only passing tokenizer as dataset now pads
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Pass the data_collator to handle padding and tensor conversion
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, collate_fn=data_collator)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)

# SANITY CHECK
sample_batch = next(iter(train_loader))

print("--- PIPELINE SANITY CHECK ---")
print("Input IDs Tensor Shape      :", sample_batch['input_ids'].shape)       # Expected: [BATCH_SIZE, MAX_LENGTH]
print("Attention Mask Tensor Shape :", sample_batch['attention_mask'].shape)  # Expected: [BATCH_SIZE, MAX_LENGTH]
print("Labels Tensor Shape         :", sample_batch['labels'].shape)          # Expected: [BATCH_SIZE]

# Verify tokenized input_ids contain actual non-zero token embeddings
if (sample_batch['input_ids'] == tokenizer.pad_token_id).all():
    print("⚠️ Warning: All input_ids in the sample batch are padding tokens. This is unusual.")
elif (sample_batch['input_ids'] == 0).any(): # Check if any input_ids are actual zero tokens (RoBERTa pad is 1)
    print("✅ Tokenization check passed: Input IDs contain non-padding/non-zero tokens.")
else:
    print("✅ Tokenization check passed: Input IDs contain non-padding tokens.")

print("\n🎉 Data Engineering Pipeline is 100% Ready for RoBERTa Fine-tuning!")

## Step 5: Hugging Face Trainer Execution

In [ ]:
import torch
import json
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# --- Model Initialization & Hugging Face Trainer Setup ---

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load id2label mapping from file to ensure availability
try:
    with open("saved_models/label_map.json", "r") as f:
        label_map_data = json.load(f)

    # The saved json has keys as strings, and values as ints. We need to convert them
    # id2label expects {int: str}, label2id expects {str: int}
    id2label = {int(k): str(v) for k, v in label_map_data['id2label'].items()}
    label2id = {str(k): int(v) for k, v in label_map_data['label2id'].items()}

except FileNotFoundError:
    raise FileNotFoundError("Error: 'label_map.json' not found. Please ensure the label mapping cell has been executed.")
except KeyError:
    raise KeyError("Error: 'id2label' or 'label2id' not found in 'label_map.json'. Check content of file.")

num_labels = len(id2label)
MODEL_NAME = "roberta-base"

# Load the model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
).to(device)

print(f"✅ {MODEL_NAME} Model successfully loaded on {device} with appropriate config!")

# Define compute_metrics function for evaluation
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    accuracy = accuracy_score(p.label_ids, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average='weighted', zero_division=0) # Changed average to 'weighted'
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# Data Collator for dynamic padding (though our dataset already pads to max_length)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Configure TrainingArguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16, # User specified: Adjusted batch size to 16
    per_device_eval_batch_size=16,  # User specified: Adjusted eval batch size to 16 for consistency
    learning_rate=3e-5,             # User specified: Changed LR to 3e-5
    weight_decay=0.01,
    warmup_ratio=0.1,               # User specified: Added warmup_ratio
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",     # User specified: Monitor F1 score for best model
    greater_is_better=True,
    logging_dir='./logs',
    logging_steps=50,
    fp16=True,                      # User specified: Enabled FP16 for T4 GPU acceleration
    gradient_accumulation_steps=1,
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
) # Removed tokenizer=tokenizer

print("\n🚀 Starting Training with Hugging Face Trainer API!")

# Train the model
trainer.train()

print("\n🎉 Model Training Completed Successfully with Hugging Face Trainer!")

# Evaluate the best model on the validation set
print("\nEvaluating best model on validation set...")
val_metrics = trainer.evaluate(eval_dataset=val_dataset)
print(f"Validation Metrics: {val_metrics}")

# Evaluate the best model on the test set
print("\nEvaluating best model on test set...")
test_metrics = trainer.evaluate(eval_dataset=test_dataset)
print(f"Test Metrics: {test_metrics}")

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU is available! You're ready to use it for training.")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is NOT available. Please go to 'Runtime' -> 'Change runtime type' and select a GPU (e.g., T4 GPU).")

## Save Trained Model Weights

In [ ]:
# Cell 13: Save Model Artifacts
SAVE_PATH = "./saved_emotion_model" # Updated save path to ./saved_emotion_model
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"✅ Trained model & tokenizer saved at '{SAVE_PATH}'!")

## Baseline Machine Learning Model Comparison

This section trains and evaluates several traditional machine learning models using TF-IDF features to establish baselines for the 7-class emotion classification task. The models will be compared based on accuracy, precision, recall, and F1-score, along with their training time.

In [ ]:
import pandas as pd
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# --- 1. Load Data ---
print("--- 1. Loading Training and Test Data ---")
train_df = pd.read_csv('saved_datasets/train.csv')
test_df = pd.read_csv('saved_datasets/test.csv')

X_train = train_df['text']
y_train = train_df['encoded_label']
X_test = test_df['text']
y_test = test_df['encoded_label']

print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")

# --- 2. Feature Extraction (TF-IDF) ---
print("\n--- 2. Performing TF-IDF Feature Extraction ---")
tfidf_vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"TF-IDF features extracted. Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")
print(f"Shape of X_train_tfidf: {X_train_tfidf.shape}")
print(f"Shape of X_test_tfidf: {X_test_tfidf.shape}")

# --- 3. Model Training and Evaluation Function ---
def train_evaluate_model(name, model, X_train, y_train, X_test, y_test):
    start_time = time.time()
    print(f"\n--- Training {name} --- ")
    model.fit(X_train, y_train)
    train_time = time.time() - start_time
    print(f"Training finished in {train_time:.2f} seconds.")

    print(f"--- Evaluating {name} ---")
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted', zero_division=0)

    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Weighted Precision: {precision:.4f}")
    print(f"  Weighted Recall: {recall:.4f}")
    print(f"  Weighted F1-Score: {f1:.4f}")

    return {
        'Model': name,
        'Accuracy': accuracy,
        'Weighted Precision': precision,
        'Weighted Recall': recall,
        'Weighted F1-Score': f1,
        'Training Time (s)': train_time
    }

# --- 4. Initialize and Evaluate Models ---
results = []

models = [
    ('Multinomial Naive Bayes', MultinomialNB()),
    ('Logistic Regression', LogisticRegression(max_iter=1000, random_state=42)),
    ('Linear SVC', LinearSVC(random_state=42, dual=False)), # dual=False for large number of samples/features
    ('Random Forest Classifier', RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)),
    ('K-Nearest Neighbors', KNeighborsClassifier(n_neighbors=5, n_jobs=-1))
]

for name, model in models:
    result = train_evaluate_model(name, model, X_train_tfidf, y_train, X_test_tfidf, y_test)
    results.append(result)

# --- 5. Comparison Summary ---
print("\n--- 5. Comparison Summary ---")
comparison_df = pd.DataFrame(results)
comparison_df_sorted = comparison_df.sort_values(by='Weighted F1-Score', ascending=False).reset_index(drop=True)

print("\nModel Performance Comparison (Sorted by Weighted F1-Score):")
display(comparison_df_sorted)

print("\n🎉 Baseline model evaluation complete!")

In [ ]:
!pip install transformers-interpret

## Part 1: Retrain RoBERTa with Target Label Fix & Part 2: Explainability & Token Heatmap (XAI)

In [ ]:
!pip install -U datasets torchvision transformers

import os
import pandas as pd
import numpy as np
import torch
from datasets import Dataset # Import Dataset from the datasets library
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers_interpret import SequenceClassificationExplainer
from IPython.display import HTML

# --- Global Configurations ---
MODEL_NAME = "roberta-base"
MAX_LENGTH = 128
SAVE_DIR = "./saved_emotion_model_retrained"

# --- Define 7-class label mapping explicitly to avoid inconsistencies ---
id2label = {
    0: "neutral",
    1: "sadness_grief",
    2: "joy_amusement_excitement_optimism",
    3: "anger_annoyance_disapproval_disgust",
    4: "desire",
    5: "fear_nervousness",
    6: "love"
}
label2id = {v: k for k, v in id2label.items()}
NUM_LABELS = len(id2label)

# --- Part 1: Retrain RoBERTa with Target Label Fix ---
print("--- Part 1: Retraining RoBERTa with Target Label Fix ---")

# 1. Load Data
print("\n1. Loading Training, Validation, and Test Data...")
train_df = pd.read_csv('saved_datasets/train.csv')
val_df = pd.read_csv('saved_datasets/validation.csv')
test_df = pd.read_csv('saved_datasets/test.csv')

# 2. CRITICAL: Rename 'encoded_label' to 'labels' strictly in all dataframes
train_df = train_df.rename(columns={'encoded_label': 'labels'})
val_df = val_df.rename(columns={'encoded_label': 'labels'})
test_df = test_df.rename(columns={'encoded_label': 'labels'})

# Convert Pandas DataFrames to Hugging Face Dataset objects
train_dataset_hf = Dataset.from_pandas(train_df[['text', 'labels']])
val_dataset_hf = Dataset.from_pandas(val_df[['text', 'labels']])
test_dataset_hf = Dataset.from_pandas(test_df[['text', 'labels']])

print(f"Loaded train dataset: {len(train_dataset_hf)} samples")
print(f"Loaded validation dataset: {len(val_dataset_hf)} samples")
print(f"Loaded test dataset: {len(test_dataset_hf)} samples")

# 3. Tokenize using roberta-base tokenizer
print("\n3. Initializing Tokenizer and Tokenizing Datasets...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding='max_length', max_length=MAX_LENGTH)

tokenized_train_dataset = train_dataset_hf.map(tokenize_function, batched=True)
tokenized_val_dataset = val_dataset_hf.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset_hf.map(tokenize_function, batched=True)

# Set format for PyTorch
tokenized_train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# 4. Train roberta-base with num_labels=7 using Trainer API
print("\n4. Initializing Model and Hugging Face Trainer...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Tokenizer and {MODEL_NAME} model initialized and moved to {device}.")

# Define compute_metrics function for evaluation
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    accuracy = accuracy_score(p.label_ids, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average='weighted', zero_division=0)
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Ensure GPU is available for fp16
fp16_enabled = torch.cuda.is_available()
if not fp16_enabled:
    print("Warning: CUDA not available. fp16 training will be disabled.")

training_args = TrainingArguments(
    output_dir='./results_retrained',
    num_train_epochs=3,
    per_device_train_batch_size=16, # User specified
    per_device_eval_batch_size=16,  # User specified
    learning_rate=2e-5,             # User specified
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_dir='./logs_retrained',
    logging_steps=50,
    fp16=fp16_enabled,
    gradient_accumulation_steps=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Hugging Face Trainer initialized. Starting training...")
trainer.train()
print("\n🎉 Model Retraining Completed Successfully with Hugging Face Trainer!")

# 5. Print test set evaluation metrics and save the best model
print("\n5. Evaluating best model on test set...")
test_metrics = trainer.evaluate(eval_dataset=tokenized_test_dataset)
print(f"Test Metrics (Accuracy & Weighted F1): {test_metrics}")

os.makedirs(SAVE_DIR, exist_ok=True);
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"✅ Retrained model & tokenizer saved at '{SAVE_DIR}'!")


# --- Part 2: Explainability & Token Heatmap (XAI) ---
print("\n--- Part 2: Explainability & Token Heatmap (XAI) ---")

# Initialize SequenceClassificationExplainer
print("\n1. Initializing SequenceClassificationExplainer...")
xai_explainer = SequenceClassificationExplainer(
    model,
    tokenizer,
    classification_tokenizer=tokenizer # Ensure the correct tokenizer is used
)

# Create a helper function explain_emotion_text
print("\n2. Defining XAI helper function `explain_emotion_text`...")
def explain_emotion_text(input_text):
    print(f"\n--- Explaining: '{input_text}' ---")

    # Get predictions first to display class and confidence
    outputs = model(tokenizer(input_text, return_tensors="pt").to(device))
    probabilities = torch.softmax(outputs.logits, dim=-1)
    predicted_label_id = torch.argmax(probabilities, dim=-1).item()
    predicted_label_name = id2label[predicted_label_id]
    confidence = probabilities[0][predicted_label_id].item()

    print(f"Predicted Emotion: {predicted_label_name} (Confidence: {confidence:.4f})")

    # Generate attributions and visualize
    word_attributions = xai_explainer(input_text)
    html_output = xai_explainer.visualize(
        word_attributions,
        predicted_label_name,
        output_type="html",
        display_html=False
    )
    display(HTML(html_output))

# 3. Run explain_emotion_text() on 3 test examples
print("\n3. Running XAI on sample test examples...")
sample_sentences = [
    "I love you so much!",
    "I am terrified of losing everything.",
    "Get away from me, I hate this!"
]

for sentence in sample_sentences:
    explain_emotion_text(sentence)

print("\n🎉 XAI analysis complete!")

In [ ]:
# Run XAI analysis on a specific custom sentence from the test set

# Select a custom sentence from the test set
custom_test_sentence = test_df['text'].iloc[0] # Taking the first sentence from the test set

print(f"Selected custom sentence from test set for XAI: '{custom_test_sentence}'")

# Run the explain_emotion_text function
explain_emotion_text(custom_test_sentence)